<div style="background:#131732;color:#fff;padding:28px 32px;border-radius:16px;font-family:Outfit,Arial,sans-serif">
  <div style="color:#A2FAA3;font-weight:700;letter-spacing:.08em;font-size:12px">DATA ENGINEERING WORKSHOP · 22 OCT 2026</div>
  <h1 style="margin:8px 0 6px;font-size:34px;color:#fff">Designing data pipelines for AI agents</h1>
  <div style="font-size:16px;color:#D6D8E0">Build a live order-status agent for delivery events, and prove that it tells the truth.</div>
</div>

### What you will have at the end
**By the last cell, each of you has an AI support agent running in this notebook.** It answers *"where is my order?"* from a live stream of delivery events, and you will be able to show, with numbers, why its answers are right.

### The problem we are solving
An AI agent answers with the **same confidence** whether its data is 2 seconds old or 2 days old. It cannot tell the difference.
So the quality of an agent is decided by the **data pipeline in front of it** which is resonsible for the context quality, not only by the model.

- **Batch still matters** for things that change slowly: menus, policies, monthly reports.
- **Streaming matters** because historically companies like DH were reliant on streaming pipelines for fraud detection or for things that change *during* a delivery (about 30 minutes): order status, courier location, refunds. But now there is one more reponsibility of these streaming pipelines which is providing fresh data for AI agents. A nightly batch job cannot answer *"what is happening right now?"*
- Agents also **take actions** (refunds, re-routing). A wrong action costs more than a wrong answer.

### Five claims we will prove today
| # | Claim | Proven in |
|---|---|---|
| 1 | Streams repeat and arrive out of order, so writes must be idempotent and ordered by event time | Parts 1, 2, 5 |
| 2 | Personal data must be removed **before** it reaches the vector store or the LLM | Part 2 |
| 3 | Similarity is not relevance: an agent needs metadata filters and time ordering | Part 3 |
| 4 | Freshness is a number you can measure, and a batch snapshot goes stale | Part 5 |
| 5 | Fresh data is not enough: the model must be grounded and checked | Parts 4, 5 |


**This session is mainly about data engineering: ingestion, cleaning, storage and retrieval. It also touches the overlap with AI engineering, where those pipelines feed an agent.**

**How to use this notebook:** run each cell top to bottom (`Shift + Enter`). Read the **What to look for** note above each cell, then look at the log lines it prints. There is nothing to solve.

> All data here is synthetic. Never paste real customer, rider or partner data into a notebook.

**FLOW**
events -> clean -> turn into numbers (embed) -> store -> search -> LLM writes answer

---
## Part 0 · Setup
We install the libraries and start a small open-source LLM (`gemma2:2b` via Ollama) **in the background**, because it takes a few minutes to download. By the time we need it in Part 4 it will be ready. **Before you run anything:** choose *Runtime > Change runtime type > T4 GPU*. On a CPU runtime each LLM answer takes more than a minute, on a GPU it takes a few seconds.

**What to look for:** every log line has a time, a stage tag and a message. You will see these tags all through the notebook:
`SETUP` · `STREAM` · `CLEAN` · `PII` · `EMBED` · `STORE` · `SEARCH` · `AGENT` · `EVAL` · `LIVE`

In [1]:
!uv add lancedb sentence-transformers polars requests

Resolved 127 packages in 1.92s                                       
Checked 84 packages in 13ms


In [8]:
import os, re, time, queue, threading, subprocess, statistics

os.environ["LANCE_LOG"] = "error"  # hide internal storage warnings

from datetime import datetime, timedelta

import lancedb
import polars as pl
import requests
from IPython.display import HTML, display
from sentence_transformers import SentenceTransformer

pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_rows(20)


def log(stage: str, msg: str) -> None:
    # One log format for the whole notebook: time | stage | message
    print(f"{datetime.now():%H:%M:%S}  {stage:<6} | {msg}")


def card(title: str, body: str, color: str = "#4629FF") -> None:
    # A small result box, so the key outcome of each part stands out
    display(HTML(
        f'<div style="border-left:6px solid {color};background:#F5F5F6;padding:12px 16px;'
        f'border-radius:8px;margin:8px 0;font-family:Arial,sans-serif;color:#131732">'
        f'<b>{title}</b><div style="margin-top:6px">{body}</div></div>'
    ))


log("SETUP", "Loading the embedding model all-MiniLM-L6-v2 (turns text into 384 numbers)")

t0 = time.perf_counter()

#HELP: An embedding model converts text or images into numerical vectors, enabling vector databases to quickly retrieve relevant information for LLMs.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

log("SETUP", f"Embedding model ready in {time.perf_counter() - t0:.1f}s")

table_lock = threading.Lock()  # one lock guards every read and write to the vector table

/Users/aniket.kapdule/projects/work/techbuddies-workshop/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 00:35:04.425000 91497 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


00:35:05  SETUP  | Loading the embedding model all-MiniLM-L6-v2 (turns text into 384 numbers)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5743.56it/s]


00:35:07  SETUP  | Embedding model ready in 1.9s


In [19]:
MODEL = "gemma2:2b"
llm_status = {"ready": False, "message": "starting"}


def _start_llm() -> None:
    # Connects to local Ollama or installs Ollama on Google Colab
    try:
        if __import__("shutil").which("apt-get") and not __import__("shutil").which("ollama"):
            llm_status["message"] = "installing Ollama"
            subprocess.run("sudo apt-get update -y && sudo apt-get install -y zstd pciutils",
                           shell=True, check=True, capture_output=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                           shell=True, check=True, capture_output=True)
            subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        llm_status["message"] = "connecting to Ollama server"
        server_ok = False
        res = None
        for _ in range(30):
            try:
                res = requests.get("http://localhost:11434/api/tags", timeout=1)
                if res.status_code == 200:
                    server_ok = True
                    break
            except requests.exceptions.RequestException:
                time.sleep(1)
        if not server_ok or res is None:
            raise RuntimeError("Ollama server not reachable at http://localhost:11434. Make sure 'ollama serve' is running.")

        models = [m.get("name") for m in res.json().get("models", [])]
        if not any(MODEL in m for m in models):
            llm_status["message"] = f"downloading {MODEL} (about 1.6 GB)"
            subprocess.run(["ollama", "pull", MODEL], check=True, capture_output=True)
        llm_status.update(ready=True, message="ready")
    except Exception as e:
        llm_status.update(ready=False, message=f"failed: {e}")


threading.Thread(target=_start_llm, daemon=True).start()
log("SETUP", f"Started downloading the LLM {MODEL} in the background. We continue meanwhile.")

01:02:07  SETUP  | Started downloading the LLM gemma2:2b in the background. We continue meanwhile.


---
## Part 1 · The stream is messy
In production, delivery events arrive on a message bus (Kafka or Pub/Sub). Here we use **13 events from the last hour** in Berlin.
They contain three problems that happen in every real stream:

1. **A duplicate:** `EVT-101` was delivered twice (the bus retried after a timeout).
2. **An out-of-order event:** `EVT-112` arrives *after* `EVT-111`, but it happened *before* it.
3. **Personal data:** `EVT-110` contains a phone number and an email in free text.

**What to look for:** the audit lines below the table count each problem before we fix anything.

In [10]:
BASE = datetime.now() - timedelta(minutes=60)


def ts(minutes: int) -> str:
    return (BASE + timedelta(minutes=minutes)).isoformat(timespec="seconds")


# (event_id, order_id, restaurant, category, minutes after BASE, raw_text), in ARRIVAL order
_rows = [
    ("EVT-101", "DH-9821", "Burger Berlin", "Delivery Delay", 15, "Courier got stuck in heavy traffic near Alexanderplatz. Estimated delay: 20 mins. Order contains hot food."),
    ("EVT-102", "DH-4412", "Sushi Express", "Missing Item", 18, "Customer received the order but the Spicy Salmon Roll was missing from the bag. Customer wants refund."),
    ("EVT-103", "DH-9821", "Burger Berlin", "Courier Update", 22, "Courier reached customer building. Intercom code #402 is not responding. Courier waiting at door."),
    ("EVT-104", "DH-4412", "Sushi Express", "Refund", 25, "Refund approved for the missing Spicy Salmon Roll."),
    ("EVT-105", "DH-7305", "Pizza Roma", "Delivery Delay", 10, "Courier stuck in heavy traffic near Alexanderplatz. Estimated delay: 25 mins."),
    ("EVT-106", "DH-7305", "Pizza Roma", "Courier Update", 20, "Courier delivered the order to the customer."),
    ("EVT-107", "DH-6150", "Pad Thai House", "Delivery Delay", 12, "Heavy congestion in Kreuzberg is slowing the courier. Estimated delay: 15 mins."),
    ("EVT-108", "DH-6150", "Pad Thai House", "Courier Update", 30, "Courier delivered the order to the customer."),
    ("EVT-109", "DH-3377", "Burger Berlin", "Missing Item", 5, "Customer reports the fries were missing from the bag."),
    ("EVT-110", "DH-5528", "Falafel Hub", "Courier Update", 16, "Courier could not reach the customer. Customer phone +49 170 1234567 and email anna.k@example.com did not respond."),
    ("EVT-111", "DH-8890", "Curry Corner", "Courier Update", 28, "Order delivered to the customer."),
    ("EVT-112", "DH-8890", "Curry Corner", "Courier Update", 8, "Courier picked up the order from the restaurant."),
    ("EVT-101", "DH-9821", "Burger Berlin", "Delivery Delay", 15, "Courier got stuck in heavy traffic near Alexanderplatz. Estimated delay: 20 mins. Order contains hot food."),
]

raw_events = [
    {"arrival_no": i + 1, "event_id": e, "order_id": o, "timestamp": ts(m), "restaurant": r, "issue_category": c, "raw_text": t}
    for i, (e, o, r, c, m, t) in enumerate(_rows)
]

df_raw = pl.DataFrame(raw_events)

log("STREAM", f"Received {df_raw.height} events from the bus, for {df_raw['order_id'].n_unique()} different orders")
display(df_raw.select("arrival_no", "event_id", "order_id", "timestamp", "issue_category", "raw_text"))

00:38:45  STREAM | Received 13 events from the bus, for 7 different orders


arrival_no,event_id,order_id,timestamp,issue_category,raw_text
i64,str,str,str,str,str
1,"""EVT-101""","""DH-9821""","""2026-10-21T23:53:45""","""Delivery Delay""","""Courier got stuck in heavy traffic near Alexanderplatz. Estimated delay: 20 mins. Order contains hot food."""
2,"""EVT-102""","""DH-4412""","""2026-10-21T23:56:45""","""Missing Item""","""Customer received the order but the Spicy Salmon Roll was missing from the bag. Customer wants refund."""
3,"""EVT-103""","""DH-9821""","""2026-10-22T00:00:45""","""Courier Update""","""Courier reached customer building. Intercom code #402 is not responding. Courier waiting at door."""
4,"""EVT-104""","""DH-4412""","""2026-10-22T00:03:45""","""Refund""","""Refund approved for the missing Spicy Salmon Roll."""
5,"""EVT-105""","""DH-7305""","""2026-10-21T23:48:45""","""Delivery Delay""","""Courier stuck in heavy traffic near Alexanderplatz. Estimated delay: 25 mins."""
6,"""EVT-106""","""DH-7305""","""2026-10-21T23:58:45""","""Courier Update""","""Courier delivered the order to the customer."""
7,"""EVT-107""","""DH-6150""","""2026-10-21T23:50:45""","""Delivery Delay""","""Heavy congestion in Kreuzberg is slowing the courier. Estimated delay: 15 mins."""
8,"""EVT-108""","""DH-6150""","""2026-10-22T00:08:45""","""Courier Update""","""Courier delivered the order to the customer."""
9,"""EVT-109""","""DH-3377""","""2026-10-21T23:43:45""","""Missing Item""","""Customer reports the fries were missing from the bag."""


In [11]:
PHONE_RE = re.compile(r"\+?\d[\d\s\-()]{7,}\d")
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")

# Audit 1: duplicates
dupes = df_raw.group_by("event_id").len().filter(pl.col("len") > 1)
for row in dupes.iter_rows(named=True):
    log("STREAM", f"Duplicate: {row['event_id']} arrived {row['len']} times")

# Audit 2: out of order (an event that arrives later but happened earlier than one already seen for the same order)
latest_seen, seen_ids = {}, set()
for r in df_raw.iter_rows(named=True):
    if r["event_id"] in seen_ids:
        continue  # already counted as a duplicate
    seen_ids.add(r["event_id"])
    prev = latest_seen.get(r["order_id"])
    if prev and r["timestamp"] < prev["timestamp"]:
        log("STREAM", f"Out of order: {r['event_id']} (happened {r['timestamp'][11:]}) arrived after "
                      f"{prev['event_id']} (happened {prev['timestamp'][11:]}) for order {r['order_id']}")
    if not prev or r["timestamp"] > prev["timestamp"]:
        latest_seen[r["order_id"]] = r

# Audit 3: personal data
for r in df_raw.unique("event_id", maintain_order=True).iter_rows(named=True):
    found = PHONE_RE.findall(r["raw_text"]) + EMAIL_RE.findall(r["raw_text"])
    if found:
        log("PII", f"{r['event_id']} contains {len(found)} personal data values (phone/email) in free text")

card("Part 1 result",
     f"{df_raw.height} events arrived, {df_raw['event_id'].n_unique()} are unique. "
     "We found 1 duplicate, 1 out-of-order event and personal data in 1 event. "
     "If we fed this straight into an agent, it would count one delay twice, could report an old status as the current one, "
     "and would put a customer's phone number into its memory.", "#D61F26")

00:38:50  STREAM | Duplicate: EVT-101 arrived 2 times
00:38:50  STREAM | Out of order: EVT-112 (happened 23:46:45) arrived after EVT-111 (happened 00:06:45) for order DH-8890
00:38:50  PII    | EVT-110 contains 2 personal data values (phone/email) in free text


---
## Part 2 · Clean the stream before the agent sees it
Three steps, in this order:

1. **Deduplicate** on `event_id`, the stable key every event carries.
2. **Scrub personal data** (phone, email) **before** embedding. A vector can leak the text it was built from, and deleting one person from an index later is painful (GDPR).
3. **Embed** each event: the text becomes 384 numbers that capture its *meaning*, so "courier stuck in traffic" and "heavy congestion slowing the courier" end up close together.

**What to look for:** the before and after of the personal data, and the first numbers of a vector.

In [12]:
def scrub_pii(text: str) -> str:
    text = EMAIL_RE.sub("[EMAIL]", text)
    return PHONE_RE.sub("[PHONE]", text)


# Step 1: deduplicate
df_unique = df_raw.unique(subset=["event_id"], keep="last", maintain_order=True)
log("CLEAN", f"Deduplicated on event_id: {df_raw.height} -> {df_unique.height} events "
             f"({df_raw.height - df_unique.height} duplicate removed)")

# Step 2: scrub personal data, and show every change we made
for r in df_unique.iter_rows(named=True):
    clean = scrub_pii(r["raw_text"])
    if clean != r["raw_text"]:
        log("PII", f"{r['event_id']} before: {r['raw_text']}")
        log("PII", f"{r['event_id']} after:  {clean}")

00:38:58  CLEAN  | Deduplicated on event_id: 13 -> 12 events (1 duplicate removed)
00:38:58  PII    | EVT-110 before: Courier could not reach the customer. Customer phone +49 170 1234567 and email anna.k@example.com did not respond.
00:38:58  PII    | EVT-110 after:  Courier could not reach the customer. Customer phone [PHONE] and email [EMAIL] did not respond.


In [13]:
def process_events(events_df: pl.DataFrame, verbose: bool = True) -> list[dict]:
    # Clean -> build text -> embed -> one record per event, ready for the vector store
    rows = list(events_df.iter_rows(named=True))
    texts = [f"Category: {r['issue_category']} | Detail: {scrub_pii(r['raw_text'])}" for r in rows]
    t0 = time.perf_counter()
    vectors = embedding_model.encode(texts, batch_size=32, normalize_embeddings=True).tolist()
    if verbose:
        log("EMBED", f"Embedded {len(texts)} events in {(time.perf_counter() - t0) * 1000:.0f} ms, "
                     f"each is a vector of {len(vectors[0])} numbers")
    return [
        {"vector": v, "text": t, "event_id": r["event_id"], "order_id": r["order_id"],
         "restaurant": r["restaurant"], "category": r["issue_category"], "timestamp": r["timestamp"]}
        for r, t, v in zip(rows, texts, vectors)
    ]


processed = process_events(df_unique)
sample = processed[0]
log("EMBED", f"Example text   : {sample['text']}")
log("EMBED", f"Example vector : [{', '.join(f'{x:.3f}' for x in sample['vector'][:6])}, ... 378 more]")

00:39:04  EMBED  | Embedded 12 events in 684 ms, each is a vector of 384 numbers
00:39:04  EMBED  | Example text   : Category: Missing Item | Detail: Customer received the order but the Spicy Salmon Roll was missing from the bag. Customer wants refund.
00:39:04  EMBED  | Example vector : [-0.064, 0.017, 0.013, 0.092, 0.022, 0.026, ... 378 more]


Now we store the vectors **together with their metadata** (order id, timestamp, category) in LanceDB, a small vector database that runs inside the notebook.

We write with an **upsert on `event_id`**: a new id inserts a row, an id we already have updates it. That makes writes **idempotent**: if the bus delivers the same event twice, we still have one row.

We also save a frozen copy called `nightly_batch`. It stands for a classic batch job that ran once and will not run again until tonight. We will come back to it in Part 5.

In [14]:
db = lancedb.connect("./delivery_hero_vector_db")
table = db.create_table("agent_memory", data=processed, mode="overwrite")
nightly_batch = db.create_table("nightly_batch", data=processed, mode="overwrite")


def upsert_events(records: list[dict]) -> None:
    # Idempotent write: same event_id updates the row, a new event_id inserts one
    with table_lock:
        (table.merge_insert("event_id")
              .when_matched_update_all()
              .when_not_matched_insert_all()
              .execute(records))


def row_count() -> int:
    with table_lock:
        return table.count_rows()


log("STORE", f"Table 'agent_memory' (live, will keep updating): {row_count()} rows")
log("STORE", f"Table 'nightly_batch' (frozen snapshot):         {nightly_batch.count_rows()} rows")

print("AGENT MEMORY")
display(pl.from_arrow(table.to_arrow()).select("event_id", "order_id", "timestamp", "category", "text").sort("timestamp"))

card("Part 2 result",
     "12 clean events are stored with their vectors and metadata. The phone number and email were replaced "
     "with [PHONE] and [EMAIL] before anything was embedded. <b>Claim 2: personal data never reached the vector store.</b>")

00:39:09  STORE  | Table 'agent_memory' (live, will keep updating): 12 rows
00:39:09  STORE  | Table 'nightly_batch' (frozen snapshot):         12 rows
AGENT MEMORY


[2026-10-21T22:39:09Z WARN  lance::dataset::write::insert] No existing dataset at /Users/aniket.kapdule/projects/work/techbuddies-workshop/delivery_hero_vector_db/agent_memory.lance, it will be created
[2026-10-21T22:39:09Z WARN  lance::dataset::write::insert] No existing dataset at /Users/aniket.kapdule/projects/work/techbuddies-workshop/delivery_hero_vector_db/nightly_batch.lance, it will be created


event_id,order_id,timestamp,category,text
str,str,str,str,str
"""EVT-109""","""DH-3377""","""2026-10-21T23:43:45""","""Missing Item""","""Category: Missing Item | Detail: Customer reports the fries were missing from the bag."""
"""EVT-112""","""DH-8890""","""2026-10-21T23:46:45""","""Courier Update""","""Category: Courier Update | Detail: Courier picked up the order from the restaurant."""
"""EVT-105""","""DH-7305""","""2026-10-21T23:48:45""","""Delivery Delay""","""Category: Delivery Delay | Detail: Courier stuck in heavy traffic near Alexanderplatz. Estimated delay: 25 mins."""
"""EVT-107""","""DH-6150""","""2026-10-21T23:50:45""","""Delivery Delay""","""Category: Delivery Delay | Detail: Heavy congestion in Kreuzberg is slowing the courier. Estimated delay: 15 mins."""
"""EVT-101""","""DH-9821""","""2026-10-21T23:53:45""","""Delivery Delay""","""Category: Delivery Delay | Detail: Courier got stuck in heavy traffic near Alexanderplatz. Estimated delay: 20 mins. Ord…"
"""EVT-110""","""DH-5528""","""2026-10-21T23:54:45""","""Courier Update""","""Category: Courier Update | Detail: Courier could not reach the customer. Customer phone [PHONE] and email [EMAIL] did no…"
"""EVT-102""","""DH-4412""","""2026-10-21T23:56:45""","""Missing Item""","""Category: Missing Item | Detail: Customer received the order but the Spicy Salmon Roll was missing from the bag. Custome…"
"""EVT-106""","""DH-7305""","""2026-10-21T23:58:45""","""Courier Update""","""Category: Courier Update | Detail: Courier delivered the order to the customer."""
"""EVT-103""","""DH-9821""","""2026-10-22T00:00:45""","""Courier Update""","""Category: Courier Update | Detail: Courier reached customer building. Intercom code #402 is not responding. Courier wait…"


---
## Part 3 · Similarity is not relevance
A customer asks: **"What is the courier doing right now for DH-9821?"**

First we search in this way which seems obvious: embed the question and return the most **similar** events. Then we search the way an agent needs: **filter** to the order in the question first, then sort **newest first**.

**What to look for:** the `order` column in the first search. How many results are even about `DH-9821`?

In [15]:
def extract_order_id(text: str) -> str | None:
    m = re.search(r"DH-\d+", text)
    return m.group(0) if m else None


def search_events(query: str, order_id: str | None = None, top_k: int = 3, latest_first: bool = True,
                  source=None, verbose: bool = True) -> list[dict]:
    source = source or table
    if order_id and not re.fullmatch(r"DH-\d+", order_id):
        raise ValueError("Invalid order_id")  # never put unvalidated text into a filter
    qv = embedding_model.encode(query, normalize_embeddings=True).tolist()
    with table_lock:
        q = source.search(qv).distance_type("cosine")
        if order_id:
            q = q.where(f"order_id = '{order_id}'", prefilter=True)  # metadata filter BEFORE vector ranking
        results = q.limit(20).to_list()
    if latest_first:
        results.sort(key=lambda r: r["timestamp"], reverse=True)  # event time decides what is "current"
    results = results[:top_k]
    if verbose:
        how = f"table {source.name}, "
        how += f"filter order_id = {order_id}, " if order_id else "no filter, "
        how += "newest first" if latest_first else "most similar first"
        log("SEARCH", f'"{query}" ({how}) -> {len(results)} results')
    return results


def show_results(results: list[dict], expected_order: str | None = None) -> None:
    for i, r in enumerate(results, 1):
        mark = "" if expected_order is None else ("  RIGHT ORDER" if r["order_id"] == expected_order else "  WRONG ORDER")
        print(f"   {i}. similarity={1 - r['_distance']:.2f}  order={r['order_id']}  time={r['timestamp'][11:]}  "
              f"{r['event_id']}: {r['text'].split('Detail: ')[1]}{mark}")


def format_context(results: list[dict]) -> str:
    return "\n".join(f"- [{r['timestamp']}] [{r['event_id']}] [{r['category']}] {r['text']} (Order: {r['order_id']})" for r in results)


question = "What is the courier doing right now for DH-9821?"
print("A) PURE VECTOR SEARCH")
show_results(search_events(question, top_k=3, latest_first=False), expected_order="DH-9821")
print()
print("B) FILTER BY ORDER, THEN NEWEST FIRST")
show_results(search_events(question, order_id=extract_order_id(question), top_k=3, latest_first=True), expected_order="DH-9821")

A) PURE VECTOR SEARCH
00:39:32  SEARCH | "What is the courier doing right now for DH-9821?" (table agent_memory, no filter, most similar first) -> 3 results
   1. similarity=0.55  order=DH-8890  time=00:06:45  EVT-111: Order delivered to the customer.  WRONG ORDER
   2. similarity=0.54  order=DH-9821  time=00:00:45  EVT-103: Courier reached customer building. Intercom code #402 is not responding. Courier waiting at door.  RIGHT ORDER
   3. similarity=0.54  order=DH-7305  time=23:58:45  EVT-106: Courier delivered the order to the customer.  WRONG ORDER

B) FILTER BY ORDER, THEN NEWEST FIRST
00:39:32  SEARCH | "What is the courier doing right now for DH-9821?" (table agent_memory, filter order_id = DH-9821, newest first) -> 2 results
   1. similarity=0.54  order=DH-9821  time=00:00:45  EVT-103: Courier reached customer building. Intercom code #402 is not responding. Courier waiting at door.  RIGHT ORDER
   2. similarity=0.48  order=DH-9821  time=23:53:45  EVT-101: Courier got stuck in he

**Why A fails:** the question is about *meaning* ("courier", "doing", "right now"), and every courier update in the table means roughly the same thing. The order number `DH-9821` is just a few characters to the embedding model. Similar is not the same as relevant.

**Why B works:** the order id and the timestamp are **metadata**, stored next to the vector. We use them as hard filters and for ordering, and let the vector only do what it is good at.

One example is an anecdote. Let us measure it on 5 questions where we know the correct answer (the newest event of each order). This small labelled set is called a **golden dataset**.

In [16]:
eval_set = [
    ("Where is the courier for DH-9821?", "EVT-103"),
    ("What is the latest on DH-4412?", "EVT-104"),
    ("What is the status of DH-7305?", "EVT-106"),
    ("What is the status of DH-8890?", "EVT-111"),  # EVT-112 arrived later but is older: it must not win
    ("What is the latest on DH-6150?", "EVT-108"),
]


def hit_rate(name: str, retriever) -> float:
    hits = 0
    for q, expected in eval_set:
        got = retriever(q)[0]["event_id"]
        hits += got == expected
        log("EVAL", f"{name:<12} {'PASS' if got == expected else 'FAIL'}  {q:<36} expected={expected} got={got}")
    return hits / len(eval_set)


naive = lambda q: search_events(q, top_k=1, latest_first=False, verbose=False)
hybrid = lambda q: search_events(q, order_id=extract_order_id(q), top_k=1, verbose=False)

score_naive = hit_rate("pure vector", naive)
print()
score_hybrid = hit_rate("filter+time", hybrid)
card("Part 3 result",
     f"Hit@1 (is the first result the right event?): pure vector <b>{score_naive:.0%}</b>, "
     f"filter + newest first <b>{score_hybrid:.0%}</b>. Note DH-8890: the out-of-order event EVT-112 did not win, "
     "because we order by when things happened, not when they arrived. <b>Claims 1 and 3.</b>")

00:39:39  EVAL   | pure vector  PASS  Where is the courier for DH-9821?    expected=EVT-103 got=EVT-103
00:39:40  EVAL   | pure vector  FAIL  What is the latest on DH-4412?       expected=EVT-104 got=EVT-103
00:39:40  EVAL   | pure vector  FAIL  What is the status of DH-7305?       expected=EVT-106 got=EVT-103
00:39:40  EVAL   | pure vector  FAIL  What is the status of DH-8890?       expected=EVT-111 got=EVT-103
00:39:40  EVAL   | pure vector  FAIL  What is the latest on DH-6150?       expected=EVT-108 got=EVT-103

00:39:40  EVAL   | filter+time  PASS  Where is the courier for DH-9821?    expected=EVT-103 got=EVT-103
00:39:40  EVAL   | filter+time  PASS  What is the latest on DH-4412?       expected=EVT-104 got=EVT-104
00:39:40  EVAL   | filter+time  PASS  What is the status of DH-7305?       expected=EVT-106 got=EVT-106
00:39:40  EVAL   | filter+time  PASS  What is the status of DH-8890?       expected=EVT-111 got=EVT-111
00:39:40  EVAL   | filter+time  PASS  What is the latest on DH-

---
## Part 4 · Meet the agent
An agent here is simple: **retrieve context, then let an LLM write the answer**. We build two:

- **Naive agent:** pure vector search, all results pasted into the prompt, "answer the question".
- **Grounded agent:** filtered and time-ordered context, split into **CURRENT STATUS** and **HISTORY**, strict rules, and it must **cite the event id** it used. A small check in code verifies the citation, so the LLM is never the only thing between the data and the customer.

**What to look for:** the context printed above each answer. Every sentence in an answer should be traceable to a line in its context. If it is not, the model made it up.

In [20]:
def wait_for_llm(timeout_s: int = 600) -> bool:
    start = time.time()
    while not llm_status["ready"] and not llm_status["message"].startswith("failed") and time.time() - start < timeout_s:
        log("SETUP", f"LLM status: {llm_status['message']} ...")
        time.sleep(10)
    log("SETUP", f"LLM status: {llm_status['message']}")
    return llm_status["ready"]


LLM_AVAILABLE = wait_for_llm()
if not LLM_AVAILABLE:
    log("SETUP", "No LLM, so the agents fall back to a template answer built from the retrieved data. Everything else works.")


def ask_llm(prompt: str) -> str:
    res = requests.post("http://localhost:11434/api/generate",
                        json={"model": MODEL, "prompt": prompt, "stream": False, "options": {"temperature": 0}},
                        timeout=120)
    res.raise_for_status()
    return res.json()["response"].strip()

01:02:31  SETUP  | LLM status: ready


In [21]:
def naive_agent(question: str, source=None, verbose: bool = True) -> str:
    results = search_events(question, top_k=3, latest_first=False, source=source, verbose=verbose)
    context = "\n".join(f"- {r['text'].split('Detail: ')[1]}" for r in results)  # naive RAG keeps only the text: no order id, no time
    if verbose:
        log("AGENT", "Naive agent context:")
        print(context)
    if not LLM_AVAILABLE:
        return f"(template) {results[0]['text'].split('Detail: ')[1]} [{results[0]['event_id']}]"
    return ask_llm(f"Answer the customer's question using these logs.\n\n{context}\n\nQuestion: {question}\nAnswer:")


def grounded_agent(question: str, source=None, verbose: bool = True) -> str:
    order_id = extract_order_id(question)
    if not order_id:
        return "I can only answer questions about a specific order. Please include the order number, for example DH-9821."
    results = search_events(question, order_id=order_id, top_k=3, source=source, verbose=verbose)
    if not results:
        return f"I don't know. There are no events for {order_id}."
    current, history = results[0], results[1:]
    if verbose:
        log("AGENT", f"Grounded agent: CURRENT STATUS = {current['event_id']} at {current['timestamp'][11:]}, "
                     f"HISTORY = {[h['event_id'] for h in history]}")
    fallback = f"Current status of {order_id}: {current['text'].split('Detail: ')[1]} [{current['event_id']}]"
    if not LLM_AVAILABLE:
        return f"(template) {fallback}"
    prompt = f"""You are a Delivery Hero customer support agent.
Rules:
- CURRENT STATUS is the truth right now. HISTORY is older and may no longer be true.
- Answer from CURRENT STATUS. Use HISTORY only to explain why something happened.
- Cite the event id you used in square brackets, for example [{current['event_id']}].
- If the logs do not answer the question, say "I don't know".
- Maximum 2 sentences.

CURRENT STATUS:
{format_context([current])}

HISTORY:
{format_context(history) or "- none"}

QUESTION: {question}
ANSWER:"""
    answer = ask_llm(prompt)
    if f"[{current['event_id']}]" not in answer:  # guardrail: the answer must be based on the current status
        log("AGENT", f"Guardrail: the answer did not cite {current['event_id']}, replacing it with the raw current status")
        return fallback
    log("AGENT", f"Guardrail: the answer cites the current status {current['event_id']}, OK")
    return answer


q = "What is the latest on DH-4412?"  # the sushi order: a missing item, then a refund
print("NAIVE AGENT")
print("   Answer:", naive_agent(q))
print()
print("GROUNDED AGENT")
print("   Answer:", grounded_agent(q))

NAIVE AGENT
01:02:38  SEARCH | "What is the latest on DH-4412?" (table agent_memory, no filter, most similar first) -> 3 results
01:02:38  AGENT  | Naive agent context:
- Courier reached customer building. Intercom code #402 is not responding. Courier waiting at door.
- Order delivered to the customer.
- Courier delivered the order to the customer.
   Answer: The latest on DH-4412 is that the order has been delivered to the customer.

GROUNDED AGENT
01:02:42  SEARCH | "What is the latest on DH-4412?" (table agent_memory, filter order_id = DH-4412, newest first) -> 2 results
01:02:42  AGENT  | Grounded agent: CURRENT STATUS = EVT-104 at 00:03:45, HISTORY = ['EVT-102']
01:02:43  AGENT  | Guardrail: the answer cites the current status EVT-104, OK
   Answer: Your Spicy Salmon Roll refund has been approved, [EVT-104].  You can now expect the refund to be reflected in your account.


**What just happened:** the naive agent was handed only the text of the most similar events, with no order id and no time, like many tutorial RAG pipelines. **None of them is about DH-4412**, the sushi order that is waiting for a refund, and it still wrote a fluent, confident answer. An LLM does not know which of its context lines are wrong, stale or about someone else, so it does not hedge. That is how agents state false facts with high confidence.

The grounded agent got the right events in the right order, and its answer is checked against the current status in code.
Next, the real test: what happens when the world changes while the customer is waiting?

---
## Part 5 · Go live
We now start a **streaming pipeline**: a background consumer reads events from a queue in **micro-batches** (up to 5 events, or every 1 second), runs the same cleaning and embedding as Part 2, and upserts them into `agent_memory`. This is a small version of Kafka plus a stream processor.

**What to look for:** the `LIVE` lines show each event's journey and how many milliseconds it took from *sent* to *searchable*. That number is **freshness**.

In [22]:
event_queue = queue.Queue()
stop_flag = threading.Event()
ingest_log = []  # one entry per event: (event_id, order_id, milliseconds from sent to searchable)


def consumer(batch_size: int = 5, max_wait_s: float = 1.0) -> None:
    # Micro-batch consumer: flush when the batch is full OR max_wait_s has passed
    batch, last_flush = [], time.perf_counter()
    while not stop_flag.is_set():
        try:
            batch.append(event_queue.get(timeout=0.2))
        except queue.Empty:
            pass
        due = time.perf_counter() - last_flush >= max_wait_s
        if batch and (len(batch) >= batch_size or due):
            try:
                upsert_events(process_events(pl.DataFrame([b["event"] for b in batch]), verbose=False))
                now = time.perf_counter()
                for b in batch:
                    ingest_log.append((b["event"]["event_id"], b["event"]["order_id"], round((now - b["sent_at"]) * 1000)))
            except Exception as e:
                print(f"Consumer error: {e}")  # keep the consumer alive
            batch, last_flush = [], time.perf_counter()
        elif not batch:
            last_flush = time.perf_counter()


threading.Thread(target=consumer, daemon=True).start()
_counter = [200]


def emit_text(order_id, category, text, restaurant="Burger Berlin", event_id=None, timestamp=None) -> str:
    # Producer: put one event on the stream (like a rider app sending an update)
    if event_id is None:
        _counter[0] += 1
        event_id = f"EVT-{_counter[0]}"
    event = {"event_id": event_id, "order_id": order_id,
             "timestamp": timestamp or datetime.now().isoformat(timespec="seconds"),
             "restaurant": restaurant, "issue_category": category, "raw_text": text}
    event_queue.put({"event": event, "sent_at": time.perf_counter()})
    log("LIVE", f"Sent      {event_id} for {order_id} (happened {event['timestamp'][11:]}): {text}")
    return event_id


def wait_ingested(event_id: str, already_seen: int = 0, timeout_s: float = 15.0) -> int | None:
    # Block until the consumer has written this event, then report the latency
    end = time.time() + timeout_s
    while time.time() < end:
        hits = [e for e in ingest_log if e[0] == event_id]
        if len(hits) > already_seen:
            log("LIVE", f"Searchable {event_id} after {hits[-1][2]} ms (deduplicate, scrub, embed, upsert)")
            return hits[-1][2]
        time.sleep(0.05)
    log("LIVE", f"{event_id} not ingested within {timeout_s}s")
    return None


log("LIVE", "Streaming consumer is running: micro-batches of up to 5 events or every 1 second")

01:06:44  LIVE   | Streaming consumer is running: micro-batches of up to 5 events or every 1 second


### 5a · The courier gives up waiting
The intercom is still broken, so the courier leaves the order with a neighbour. We send that event and ask the same question **before** and **after**, to two agents:

- **Batch agent:** reads `nightly_batch`, the snapshot from Part 2 that will not refresh until tonight.
- **Streaming agent:** reads `agent_memory`, which the consumer keeps updating.

Both use the same grounded logic. The only difference is the data pipeline.

In [23]:
q = "What is the courier doing right now for DH-9821?"
log("AGENT", f"BEFORE the new event, streaming agent: {grounded_agent(q, verbose=False)}")
print()
live_eid = emit_text("DH-9821", "Courier Update", "Intercom is broken. Courier is leaving the order with a neighbour.")
latency_ms = wait_ingested(live_eid)
print()
batch_answer = grounded_agent(q, source=nightly_batch)
print("   BATCH agent answer    :", batch_answer)
print()
stream_answer = grounded_agent(q)
print("   STREAMING agent answer:", stream_answer)

card("Part 5a result",
     f"The new event was searchable after <b>{latency_ms} ms</b>. The streaming agent answers from {live_eid}; "
     "the batch agent still says the courier is waiting at the door, with full confidence, until tonight's job runs. "
     "<b>Claim 4: freshness is measurable, and a snapshot goes stale.</b>")

01:06:49  AGENT  | Guardrail: the answer cites the current status EVT-103, OK
01:06:49  AGENT  | BEFORE the new event, streaming agent: The courier is waiting at the customer's door.  They are waiting for the Intercom code #402 to respond.  [EVT-103]

01:06:49  LIVE   | Sent      EVT-201 for DH-9821 (happened 01:06:49): Intercom is broken. Courier is leaving the order with a neighbour.
01:06:51  LIVE   | Searchable EVT-201 after 1250 ms (deduplicate, scrub, embed, upsert)

01:06:51  SEARCH | "What is the courier doing right now for DH-9821?" (table nightly_batch, filter order_id = DH-9821, newest first) -> 2 results
01:06:51  AGENT  | Grounded agent: CURRENT STATUS = EVT-103 at 00:00:45, HISTORY = ['EVT-101']
01:06:52  AGENT  | Guardrail: the answer cites the current status EVT-103, OK
   BATCH agent answer    : The courier is waiting at the customer's door.  They are waiting for the Intercom code #402 to respond.  [EVT-103]

01:06:52  SEARCH | "What is the courier doing right now for 

### 5b · The bus sends the same event twice
Retries are normal in streaming. We send a new event `EVT-900`, then send **exactly the same event again**.

**What to look for:** the row count. Thanks to the upsert on `event_id`, the duplicate must not create a second row.

In [24]:
before = row_count()
emit_text("DH-4412", "Courier Update", "Courier is 3 minutes away with the replacement item.",
          restaurant="Sushi Express", event_id="EVT-900")
wait_ingested("EVT-900")
after_first = row_count()
log("STORE", f"Rows: {before} -> {after_first} after the first send")

emit_text("DH-4412", "Courier Update", "Courier is 3 minutes away with the replacement item.",
          restaurant="Sushi Express", event_id="EVT-900")
wait_ingested("EVT-900", already_seen=1)
after_second = row_count()
log("STORE", f"Rows: {after_first} -> {after_second} after the duplicate")

idempotent_ok = after_first == before + 1 and after_second == after_first
card("Part 5b result",
     f"Rows before / after first send / after duplicate: <b>{before} / {after_first} / {after_second}</b>. "
     + ("The duplicate was absorbed. <b>Claim 1: writes are idempotent.</b>" if idempotent_ok else "Upsert is NOT idempotent, check the pipeline."))

01:07:04  LIVE   | Sent      EVT-900 for DH-4412 (happened 01:07:04): Courier is 3 minutes away with the replacement item.
01:07:06  LIVE   | Searchable EVT-900 after 1491 ms (deduplicate, scrub, embed, upsert)
01:07:06  STORE  | Rows: 13 -> 14 after the first send
01:07:06  LIVE   | Sent      EVT-900 for DH-4412 (happened 01:07:06): Courier is 3 minutes away with the replacement item.
01:07:07  LIVE   | Searchable EVT-900 after 1076 ms (deduplicate, scrub, embed, upsert)
01:07:07  STORE  | Rows: 14 -> 14 after the duplicate


### 5c · A late event arrives
A rider's phone was offline, so an event from **40 minutes ago** ("courier picked up the order") arrives now.
If the pipeline used *arrival time*, the agent would suddenly say the courier is back at the restaurant.

**What to look for:** which event the grounded agent treats as CURRENT STATUS.

In [25]:
old_ts = (datetime.now() - timedelta(minutes=40)).isoformat(timespec="seconds")
late_eid = emit_text("DH-9821", "Courier Update", "Courier picked up the order from the restaurant.", timestamp=old_ts)
wait_ingested(late_eid)
print()
show_results(search_events("latest courier status for DH-9821", order_id="DH-9821", top_k=3))
print()
late_answer = grounded_agent(q)
print("   STREAMING agent answer:", late_answer)
late_ok = late_eid not in late_answer
card("Part 5c result",
     ("The late event was stored as history, and the current status is still " + live_eid +
      ". <b>Claim 1: order by event time, not arrival time.</b>") if late_ok else "The late event became the current status, check the ordering.")

01:09:24  LIVE   | Sent      EVT-202 for DH-9821 (happened 00:29:24): Courier picked up the order from the restaurant.
01:09:25  LIVE   | Searchable EVT-202 after 1412 ms (deduplicate, scrub, embed, upsert)

01:09:25  SEARCH | "latest courier status for DH-9821" (table agent_memory, filter order_id = DH-9821, newest first) -> 3 results
   1. similarity=0.56  order=DH-9821  time=01:06:49  EVT-201: Intercom is broken. Courier is leaving the order with a neighbour.
   2. similarity=0.48  order=DH-9821  time=00:29:24  EVT-202: Courier picked up the order from the restaurant.
   3. similarity=0.55  order=DH-9821  time=00:00:45  EVT-103: Courier reached customer building. Intercom code #402 is not responding. Courier waiting at door.

01:09:25  SEARCH | "What is the courier doing right now for DH-9821?" (table agent_memory, filter order_id = DH-9821, newest first) -> 3 results
01:09:25  AGENT  | Grounded agent: CURRENT STATUS = EVT-201 at 01:06:49, HISTORY = ['EVT-202', 'EVT-103']
01:09:26  

---
## Part 6 · Proof
At the start we made five claims. Here is the evidence, computed from what this notebook just did, not typed by hand.

In [26]:
with table_lock:
    stored_texts = table.to_arrow()["text"].to_pylist()
pii_leaks = sum(len(PHONE_RE.findall(t)) + len(EMAIL_RE.findall(t)) for t in stored_texts)
latencies = [e[2] for e in ingest_log]

rows = [
    ("1", "Duplicates and late events are handled",
     f"{df_raw.height - df_unique.height} duplicate removed in batch, duplicate re-send added {after_second - after_first} rows, "
     f"late event became current: {not late_ok}", idempotent_ok and late_ok),
    ("2", "No personal data in the vector store",
     f"{pii_leaks} phone numbers or emails found in {len(stored_texts)} stored texts", pii_leaks == 0),
    ("3", "Filters and time beat pure similarity",
     f"Hit@1: pure vector {score_naive:.0%}, filter + newest first {score_hybrid:.0%}", score_hybrid > score_naive),
    ("4", "Freshness is measurable, batch goes stale",
     f"median {statistics.median(latencies):.0f} ms from sent to searchable; batch agent cited {live_eid}: {live_eid in batch_answer}",
     live_eid not in batch_answer and live_eid in stream_answer),
    ("5", "Answers are grounded and checked",
     f"streaming agent cited the current status {live_eid}: {live_eid in stream_answer}", live_eid in stream_answer),
]

html = ['<table style="border-collapse:collapse;font-family:Arial,sans-serif;color:#131732;width:100%">',
        '<tr style="background:#131732;color:#fff"><th style="padding:8px">#</th><th style="padding:8px;text-align:left">Claim</th>'
        '<th style="padding:8px;text-align:left">Evidence from this run</th><th style="padding:8px">Result</th></tr>']
for n, claim, evidence, ok in rows:
    badge = ('<span style="background:#A2FAA3;padding:2px 10px;border-radius:999px">PROVEN</span>' if ok
             else '<span style="background:#FBE4E5;padding:2px 10px;border-radius:999px">NOT PROVEN</span>')
    html.append(f'<tr style="border-bottom:1px solid #EAEAEC"><td style="padding:8px;text-align:center"><b>{n}</b></td>'
                f'<td style="padding:8px">{claim}</td><td style="padding:8px">{evidence}</td><td style="padding:8px;text-align:center">{badge}</td></tr>')
html.append("</table>")
display(HTML("".join(html)))

#,Claim,Evidence from this run,Result
1,Duplicates and late events are handled,"1 duplicate removed in batch, duplicate re-send added 0 rows, late event became current: False",PROVEN
2,No personal data in the vector store,0 phone numbers or emails found in 15 stored texts,PROVEN
3,Filters and time beat pure similarity,"Hit@1: pure vector 20%, filter + newest first 100%",PROVEN
4,"Freshness is measurable, batch goes stale",median 1331 ms from sent to searchable; batch agent cited EVT-201: False,PROVEN
5,Answers are grounded and checked,streaming agent cited the current status EVT-201: True,PROVEN


### Your agent, live
The pipeline is still running. This is the agent we promised at the start: it answers from a live stream, ignores duplicates and late events, keeps personal data out, and cites its source.

In [27]:
for question in [
    "What is the courier doing right now for DH-9821?",
    "Did the customer of DH-4412 get their replacement?",
    "Where is my pizza?",
]:
    log("AGENT", f"Q: {question}")
    print("   A:", grounded_agent(question, verbose=False))
    print()

01:27:33  AGENT  | Q: What is the courier doing right now for DH-9821?
01:27:36  AGENT  | Guardrail: the answer cites the current status EVT-201, OK
   A: The courier is leaving the order with a neighbor because Intercom is broken. [EVT-201]

01:27:36  AGENT  | Q: Did the customer of DH-4412 get their replacement?
01:27:38  AGENT  | Guardrail: the answer cites the current status EVT-900, OK
   A: Yes, the customer of DH-4412 received their replacement item. [EVT-900]

01:27:38  AGENT  | Q: Where is my pizza?
   A: I can only answer questions about a specific order. Please include the order number, for example DH-9821.



---
## From this notebook to production
Every cell you ran has a production counterpart. The ideas stay the same, the scale changes.

| Notebook | Production | What changes at scale |
|---|---|---|
| `emit_text` + `queue.Queue` | Kafka or Pub/Sub topic | Ordering, replay, schema registry, many producers |
| `consumer` micro-batches | Flink, Dataflow or Spark Structured Streaming | Checkpoints, backpressure, exactly-once effect through idempotent keys |
| `unique("event_id")` + `merge_insert` | Idempotent upsert on a business key | Dedupe state with a time limit for very late events |
| Event-time ordering | Event time and watermarks in the stream processor | How long to wait for late events |
| `scrub_pii` regex | Cloud DLP or Presidio, plus data contracts | Measure detection recall, block the pipeline on leaks |
| `all-MiniLM-L6-v2` | Vertex AI embeddings | Batching, cost, model versioning, re-embed on upgrade |
| LanceDB table | Vertex Vector Search, BigQuery VECTOR_SEARCH or pgvector | Filter pushdown, index refresh lag |
| `extract_order_id` | LLM tool calling or structured output | Validate every extracted filter |
| Guardrail citation check | Output validation and policy checks | Never let the model be the only check |
| `eval_set` + Hit@1 | Golden datasets and Langfuse experiments in CI | Bigger labelled sets, regression gates |
| Batch snapshot | Nightly batch jobs | Still right for slow-changing knowledge: menus, policies, reports |

### Take-aways
1. **An agent is only as good as the context we give it.** It will not tell you the data is stale.
2. **Stream what changes during the customer's journey, batch what does not.**
3. **Clean in the pipeline:** deduplicate, order by event time, scrub personal data before embedding.
4. **Retrieve with metadata, not only similarity.**
5. **Prove it with numbers:** a golden dataset and a freshness metric, before and after every change.

In [ ]:
stop_flag.set()
log("LIVE", "Streaming consumer stopped. Thanks for joining!")